# Tutorial 3 — Analysing Dataset Quality Before Training

**Scenario:** Your team has been labelling a pedestrian detection dataset
for three weeks.  Before you kick off a week-long training run, your
tech lead asks:

> *"How balanced are the classes? Are there images with no annotations
> at all? How do bounding-box sizes vary across the dataset?"*

You could write a bunch of nested loops over `load_dataset()` — but
for larger datasets that becomes slow.  Instead you'll pull the data
out as **Apache Arrow record batches** (zero-copy from the dman DB)
and analyse it with Pandas and matplotlib.

**Prerequisites:** `pip install dman pillow pandas matplotlib pyarrow`

## Setup

In [ ]:
import os
import tempfile
import pathlib
import subprocess
import random

tutorial_dir = pathlib.Path(tempfile.mkdtemp(prefix="dman_tutorial3_"))
os.environ["DMAN_HOME"] = str(tutorial_dir / "catalog")

subprocess.run(["dman", "init"], check=True)
print("Ready. Working in:", tutorial_dir)

## Step 1 — Build a realistic-ish dataset

We simulate a dataset that has the kinds of problems you'd actually
find in practice:

- **Class imbalance** — many more pedestrians than cyclists
- **Unlabelled images** — some frames were skipped by the annotators
- **Size variation** — distant pedestrians have small boxes;
  close-up ones are large

In [ ]:
from PIL import Image
import dman

random.seed(7)

IMG_W, IMG_H = 320, 240
images_dir = tutorial_dir / "frames"
images_dir.mkdir()

# Simulate 60 frames; 10 of them are deliberately unlabelled (annotator skipped)
NUM_FRAMES   = 60
SKIP_FRAMES  = {4, 11, 23, 31, 38, 45, 50, 53, 57, 59}  # unlabelled

# Class distribution: pedestrian is dominant
CLASS_WEIGHTS = {"pedestrian": 0.70, "cyclist": 0.20, "vehicle": 0.10}

builder = dman.create_dataset("pedestrian_detection")
builder.set_category("pedestrian", supercategory="person")
builder.set_category("cyclist",    supercategory="person")
builder.set_category("vehicle",    supercategory="object")

for i in range(NUM_FRAMES):
    stem = f"frame_{i:04d}"
    img_path = images_dir / f"{stem}.jpg"

    # Simple grey frame — brightness varies to simulate time-of-day
    brightness = random.randint(60, 200)
    Image.new("RGB", (IMG_W, IMG_H), (brightness, brightness, brightness)).save(img_path)

    builder.add_image(str(img_path))

    if i in SKIP_FRAMES:
        continue  # deliberately no annotations

    # 1–4 objects per frame
    n_objects = random.randint(1, 4)
    for _ in range(n_objects):
        # Pick class according to weights
        cls = random.choices(
            list(CLASS_WEIGHTS.keys()),
            weights=list(CLASS_WEIGHTS.values())
        )[0]

        # Simulate distance: 'far' pedestrians have small bboxes
        if cls == "pedestrian" and random.random() < 0.4:
            bw = random.uniform(0.02, 0.06)  # far away
            bh = random.uniform(0.04, 0.12)
        else:
            bw = random.uniform(0.06, 0.30)
            bh = random.uniform(0.10, 0.50)

        cx = random.uniform(bw / 2, 1 - bw / 2)
        cy = random.uniform(bh / 2, 1 - bh / 2)

        # Convert normalised → absolute pixels
        x = int((cx - bw / 2) * IMG_W)
        y = int((cy - bh / 2) * IMG_H)
        w = int(bw * IMG_W)
        h = int(bh * IMG_H)

        builder.add_annotation(stem, category=cls,
                                bbox={"x": x, "y": y, "width": w, "height": h})

ds = builder.build()
print(f"Dataset ready: {ds.sample_count()} samples, "
      f"{ds.annotation_count()} annotations")

## Step 2 — Pull data into Pandas via Arrow

`ds.to_arrow()` returns four zero-copy `pyarrow.RecordBatch` objects.
We convert them to Pandas DataFrames for analysis.

**Zero-copy** means the data isn't serialised or duplicated — the
Python process reads directly from the same memory the Rust DB layer
populated.  For large datasets this matters a lot.

In [ ]:
import pandas as pd
import pyarrow as pa

arrow = ds.to_arrow()

samples_df     = arrow["samples"].to_pandas()
annotations_df = arrow["annotations"].to_pandas()
categories_df  = arrow["categories"].to_pandas()

print("samples columns    :", list(samples_df.columns))
print("annotations columns:", list(annotations_df.columns))
print("categories columns :", list(categories_df.columns))

## Step 3 — Find unlabelled images

An unlabelled image will silently drag down recall — the model sees
the object but gets no positive supervision signal.  Let's identify them.

In [ ]:
labelled_ids = set(annotations_df["sample_id"].unique())
all_ids      = set(samples_df["id"].unique())
unlabelled   = samples_df[~samples_df["id"].isin(labelled_ids)]

print(f"Total samples  : {len(samples_df)}")
print(f"Labelled       : {len(labelled_ids)}")
print(f"Unlabelled     : {len(unlabelled)}")
print()
print("Unlabelled sample names:")
for name in sorted(unlabelled["name"]):
    print(" ", name)

## Step 4 — Class balance

A model trained on imbalanced data will be biased toward the majority
class.  Knowing the imbalance ratio now lets you decide whether to
oversample, undersample, or adjust loss weights.

In [ ]:
import matplotlib.pyplot as plt

# Join category names onto annotations
ann = annotations_df.merge(
    categories_df[["id", "name"]].rename(columns={"id": "category_id", "name": "category_name"}),
    on="category_id",
    how="left",
)

class_counts = ann["category_name"].value_counts()

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(class_counts.index, class_counts.values,
              color=["steelblue", "coral", "seagreen"])
ax.bar_label(bars)
ax.set_title("Annotation count per class")
ax.set_ylabel("Count")
ax.set_xlabel("Class")
plt.tight_layout()
plt.show()

majority = class_counts.max()
minority = class_counts.min()
print(f"\nImbalance ratio (majority / minority): {majority / minority:.1f}×")

## Step 5 — Bounding-box size distribution

Very small boxes (< 16×16 px) are notoriously hard for models to
detect.  Let's see how much of our data falls in that danger zone.

In [ ]:
import json
import numpy as np

# Parse the JSON bbox strings into columns
bboxes = ann["bbox"].dropna().apply(json.loads)
ann = ann.copy()
ann["bw"] = bboxes.apply(lambda b: b["width"])
ann["bh"] = bboxes.apply(lambda b: b["height"])
ann["area"] = ann["bw"] * ann["bh"]

# Scatter plot: width vs height, coloured by class
COLORS = {"pedestrian": "steelblue", "cyclist": "coral", "vehicle": "seagreen"}

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: scatter
for cls, grp in ann.groupby("category_name"):
    axes[0].scatter(grp["bw"], grp["bh"], alpha=0.5,
                    label=cls, color=COLORS.get(cls, "grey"), s=20)
axes[0].axvline(16, color="red", linestyle="--", linewidth=0.8, label="16 px threshold")
axes[0].axhline(16, color="red", linestyle="--", linewidth=0.8)
axes[0].set_xlabel("Box width (px)")
axes[0].set_ylabel("Box height (px)")
axes[0].set_title("Bbox size scatter")
axes[0].legend(fontsize=7)

# Right: area histogram
axes[1].hist(ann["area"], bins=30, color="slategrey", edgecolor="white")
axes[1].axvline(16 * 16, color="red", linestyle="--", linewidth=0.8,
                label="16×16 px² threshold")
axes[1].set_xlabel("Box area (px²)")
axes[1].set_ylabel("Count")
axes[1].set_title("Bbox area distribution")
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

tiny = ann[ann["area"] < 16 * 16]
print(f"Annotations with area < 16×16 px: {len(tiny)} "
      f"({100 * len(tiny) / len(ann):.1f}% of all annotations)")
print("By class:")
print(tiny["category_name"].value_counts().to_string())

## Step 6 — Annotations per image distribution

Frames with many objects are harder to train on and can slow down
data loading if they're unexpectedly large.  A quick histogram
reveals outliers.

In [ ]:
ann_per_sample = ann.groupby("sample_id").size().reset_index(name="count")

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(ann_per_sample["count"], bins=range(1, ann_per_sample["count"].max() + 2),
        color="steelblue", edgecolor="white", align="left")
ax.set_xlabel("Annotations per image")
ax.set_ylabel("Number of images")
ax.set_title("How crowded are the frames?")
ax.set_xticks(range(1, ann_per_sample["count"].max() + 1))
plt.tight_layout()
plt.show()

print(ann_per_sample["count"].describe().to_string())

## Step 7 — Summary report

Package the findings into a dict you can log, print, or attach
to an experiment tracker.

In [ ]:
report = {
    "total_samples":        int(ds.sample_count()),
    "labelled_samples":     int(len(labelled_ids)),
    "unlabelled_samples":   int(len(unlabelled)),
    "total_annotations":    int(ds.annotation_count()),
    "class_counts":         class_counts.to_dict(),
    "imbalance_ratio":      round(float(majority / minority), 2),
    "tiny_bbox_count":      int(len(tiny)),
    "tiny_bbox_pct":        round(100 * len(tiny) / len(ann), 1),
    "avg_annotations_per_labelled_image": round(
        float(ann_per_sample["count"].mean()), 2
    ),
}

print("Dataset quality report:")
print("-" * 40)
for k, v in report.items():
    print(f"  {k:<40} {v}")

# --- decision gate ---
issues = []
if report["unlabelled_samples"] > 0:
    issues.append(f"{report['unlabelled_samples']} unlabelled images — review or exclude them")
if report["imbalance_ratio"] > 5:
    issues.append(f"Imbalance ratio {report['imbalance_ratio']}× — consider oversampling minority classes")
if report["tiny_bbox_pct"] > 10:
    issues.append(f"{report['tiny_bbox_pct']}% tiny bboxes — consider a multi-scale detector")

print()
if issues:
    print("Issues to address before training:")
    for issue in issues:
        print(" ⚠", issue)
else:
    print("No major issues detected — dataset looks ready.")

## Cleanup

In [ ]:
import shutil
shutil.rmtree(tutorial_dir)
print("Cleaned up:", tutorial_dir)

---

**What you learned:**

- `ds.to_arrow()` exports all tables as zero-copy `pyarrow.RecordBatch` objects
- Converting to Pandas lets you use the full data-analysis ecosystem
- Concrete checks to run before training: unlabelled images, class balance, bbox size distribution

**Next steps:**
- Fix unlabelled images using Label Studio (`dman label-studio`)
- Export the cleaned dataset with `dman export --format huggingface` and start training